# Exercise 3: Topic Modeling 

We will walk through the basic pipeline to build and interpret a topic model, including understanding key evaluation metrics.  

Goal: Understanding the basics of topic modeling. 




In [ ]:
from pathlib import Path
import sys

sys.path.append("../")

# Check imports
from src.config import REPO

# Setup

In [ ]:
from pathlib import Path
import logging
import shutil
import sys
import json

sys.path.append("../")

# Check imports
from src.config import REPO
from solutions.data_models import PostDocument
from solutions.preprocess import PreprocessingPipeline as PreprocessingPipelineSolution

# Make sure we have preprocessed posts and have the necessary data files.
# We will load preprocessed data files from the solutions directory to save
# time.

def load_preprocessed_data() -> list[PostDocument]:
    source_path = REPO / "solutions" / "processed_posts_novision.json.bkp"
    if not source_path.exists():
        raise FileNotFoundError(f"Missing source file: {source_path}")
    with open(source_path, "r") as f:
        data = json.load(f)
    raw_items = data.values()
    return [PostDocument(**item) for item in raw_items]

def run_truncated_pipeline():
    # Load preprocessed data (without vision features)
    postdocs = load_preprocessed_data()
    # Store results: clear existing outputs, then save to Elasticsearch (if
    # available) and disk.
    preprocesser = PreprocessingPipelineSolution()
    preprocesser.clear_stored_outputs()
    preprocesser.save_to_elasticsearch(postdocs)
    preprocesser.save_processed_posts(postdocs)

run_truncated_pipeline()

# Training the topic model can be time-consuming, so we run it first with
# default parameters while we are
# reviewing the code in this notebook. Later, you will experiment with changing
# the parameters to see how it affects the resulting topics.
from src.topic_model import TopicModeler
TopicModeler().run()

## Outline

To save time: 
Run src/topic_model.py to train the model. 

While the model is running we will: 

1. Step by step code for building a typical topic model pipeline using BERTopic. At
each step the pipeline the notebook will include information on the following: 
- The model being called.
- What does does and why it is included.
- An explanation of the underlying mechanics and parameters passed to the model.
- Links to documentation 

Once the model is trained, view topics in the Demo App. 

Then return to this notebook again to evaluate the results using the information
stored in output/

2. Reviewing topics 
    - Mapping topic clusters back to the posts assigned to the cluster. 
    - What is returned in "-1"
    - How are topics labeled? 

2. Evaluating the model key metrics. 
    - intertopic distance
    - intratopic distance 
    - examining word probabilities 
    - Using c-TFI-DF to see which words are contributing the most to the topic cluster



# 1. Building a topic training pipeline (using BERTopic)

### The BERTopic pipeline at a glance

BERTopic is a five-stage pipeline.  At each stage you can choose to use a
different model (i.e. K-Means not HDBSCAN for clustering)

```
  documents → pre-computed embeddings → UMAP → HDBSCAN → CountVectorizer → c-TF-IDF → KeyBERT/LLM
             (any model)  (reduce) (cluster) (vocabulary)    (re-weight)  (label)
```

1. **Embed** each document (we already did this in Notebook 2 — 384-d vectors).
2. **UMAP** projects the 384-d embeddings down to ~2–10 dimensions so that clustering is
   meaningful (HDBSCAN struggles in high dimensions).
3. **HDBSCAN** finds dense regions in the reduced space; everything else is labelled `-1`
   ("outlier").
4. **CountVectorizer + c-TF-IDF** treats each topic-cluster as one big "class document"
   and finds the n-grams that best distinguish it from the other classes.
5. **KeyBERT / LLM** turn those keywords into a human-readable label. (Referred
   to as "representation")

Reference: https://maartengr.github.io/BERTopic/algorithm/algorithm.html

### Step 0 — Load pre-computed embeddings

Re-embedding all posts during the workshop is slow. Instead we read the embeddings
produced by `src/preprocess.py` from `output/processed_posts.json`.

In [ ]:
import json

import numpy as np

from src.config import OUTPUT
from src.data_models import PostDocument
from src.preprocess import extract_embedding_text

with open(OUTPUT / "processed_posts.json") as f:
    raw = json.load(f)

postdocs = [PostDocument(**postdoc) for postdoc in raw.values() if postdoc.get("doc_embedding")]
texts = [extract_embedding_text(postdoc) for postdoc in postdocs]
embeddings = np.array([postdoc.doc_embedding for postdoc in postdocs], dtype=np.float32)

print(f"Loaded {len(texts)} documents.")
print(f"Embeddings shape: {embeddings.shape}  (one row per doc, 384 dims)")

### Step 1 — UMAP: dense, contextual → low-dimensional, separable

Why reduce? In 384-d space, distances between *all* points are large and similar — the
"curse of dimensionality". HDBSCAN works much better on a 2–10 dim projection that
preserves *local* structure.

Key parameters:
- `n_neighbors` — how much *local* vs *global* structure to preserve. Smaller = more
  local detail, more clusters.
- `n_components` — target dimensionality of the UMAP output. Lower values simplify
  structure for clustering; higher values preserve more nuance.
- `min_dist` — how tightly packed neighbours can be in the projection. `0.0` lets
  clusters collapse to points (good for HDBSCAN).
- `metric="cosine"` — distance metric used to build neighborhoods in the original
  embedding space. Matches how are embeddings were generated.
- `output_metric="euclidean"` — distance metric of the reduced space UMAP outputs;
  this is what HDBSCAN should use downstream.

IMPORTANT: We use "cosine" distance on the raw embeddings to measure the
difference in angles between the vectors. UMAP will reduce the dimensionality
and output vectors that are designed to be measured using "euclidean" distance.
So, when we set the distance metric for HDBSCAN we use "euclidean" distance. 

Reference: https://umap-learn.readthedocs.io/en/latest/parameters.html

In [ ]:
from umap import UMAP

# Note: Try n-neighbors=5 for more local structure, n_neighbors=50 for more global structure.
RANDOM_SEED = 99
umap_model = UMAP(
    n_neighbors=15,  # local vs global balance
    n_components=2,  # output dimensions
    metric="euclidean",  # distance in input space
    output_metric="euclidean",  # distance in reduced space
    min_dist=0.1,  # minimum spacing in projection
    random_state=RANDOM_SEED,  # reproducibility
)
umap_embeddings = umap_model.fit_transform(embeddings)
print(f"UMAP-reduced embeddings shape: {umap_embeddings.shape}  (was {embeddings.shape})")
print(f"First 3 reduced points:\n{umap_embeddings[:3]}")

### Step 2 — HDBSCAN: density-based clustering

HDBSCAN finds regions in the UMAP space that are dense enough to be a cluster and
labels everything else as `-1` (outliers). Unlike k-means, you do **not** specify the
number of clusters — the data does.

Key parameters:
- `min_cluster_size` — the smallest number of points that can form a cluster. Larger →
  fewer, broader topics; smaller → more, more specific topics.
- `min_samples` — how conservative the algorithm is. Larger → more outliers.

Reference: https://hdbscan.readthedocs.io/en/latest/parameter_selection.html

In [ ]:
from collections import Counter

from hdbscan import HDBSCAN

#Note: Try min_cluster_size=5 for more clusters, min_cluster_size=20 for fewer
#clusters. Also, min_samples controls how conservative the clustering is. Higher values
#will label more points as outliers (cluster label -1).

hdbscan_model = HDBSCAN(
    min_cluster_size=5,  # smallest cluster size
    min_samples=None,  # None -> uses min_cluster_size
    metric="euclidean",  # distance in UMAP space
    cluster_selection_method="eom",  # picks the most stable persistent clusters across density levels. Alternative: 'leaf' picks the smallest clusters at the leaves of the cluster hierarchy.
    allow_single_cluster=False,  # avoid one giant cluster
    prediction_data=True,  # needed for BERTopic probabilities
)
cluster_labels = hdbscan_model.fit_predict(umap_embeddings)
counts = Counter(cluster_labels)
print("Cluster sizes (label -1 = outliers):")
for label, n in sorted(counts.items()):
    print(f"  topic {label:3d}: {n:4d} documents")

### Step 3 — CountVectorizer: build the topic vocabulary

Once we have clusters, we still need *words* to describe them. BERTopic concatenates
every document in a cluster into one big "class document" and then runs a standard
`CountVectorizer` to build a vocabulary.

Key parameters (slide 30 — vocabulary pruning):
- `min_df` — drop terms that appear in fewer than this many topics (kills typos
  and one-offs).
- `max_df` — drop terms that appear in more than this fraction of topics
  (kills very common, near-stop-words).
- `ngram_range=(1, 3)` — keep unigrams, bigrams, and trigrams. Better handles
  words that appear in phrases. 
- `stop_words="english"` — drop the standard English stop-list.

IMPORTANT: Because BERTopic feeds all the terms that appear in a topic to
CountVectorizer as one document, change `min_df` and `max_df` can  significantly 
change the topic keywords and labels.

Reference: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# Used for topic word extraction after clustering.
# Note: Try min_df=2 to ignore very rare words, max_df=0.5 to ignore very common
# words, ngram_range=(1,2) to include bigrams (e.g. "machine learning"), etc.
# See how the topic keywords and coherence change.
vectorizer = CountVectorizer(
    input="content",  # input is raw text content; alternately load a file.
    encoding="utf-8",  # text encoding
    decode_error="strict",  # fail on bad decode
    strip_accents=None,  # keep accents
    lowercase=True,  # lowercase tokens
    preprocessor=None,  # optional custom preprocessor
    tokenizer=None,  # optional custom tokenizer
    stop_words=None,  # no stop-word list
    token_pattern=r"(?u)\b\w\w+\b",  # token regex (2+ chars)
    ngram_range=(1, 1),  # unigrams only; adjust for bigrams ("music vibes"), trigrams ("high speed rail"), etc.
    analyzer="word",  # analyze at word level
    max_df=1.0,  # keep very common terms
    min_df=1,  # keep rare terms
    max_features=None,  # no vocab cap
    vocabulary=None,  # learn vocabulary from corpus
    binary=False,  # use term counts, not binary
    dtype=np.int64,  # matrix integer type
)
X = vectorizer.fit_transform(texts)
print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")
print(f"Document-term matrix shape: {X.shape}")
print("Sample n-grams:", list(vectorizer.vocabulary_.keys())[:10])

### Step 4 — c-TF-IDF: which words distinguish *this* topic from the others?

Plain TF-IDF compares a document against the rest of the corpus. **Class-based**
TF-IDF (c-TF-IDF) compares one *class* (topic) against the rest of the *classes*.
Result: the highest-scoring terms for a topic are the ones that are common *inside*
the topic but rare *outside* of it. That is the per-topic keyword ranking BERTopic
shows you.

Reference: https://maartengr.github.io/BERTopic/api/ctfidf.html

In [ ]:
from bertopic.vectorizers import ClassTfidfTransformer

# Demo: build the per-cluster "class documents" and re-weight them with c-TF-IDF.
ctfidf = ClassTfidfTransformer()
# Group docs by HDBSCAN cluster label
from collections import defaultdict
cluster_docs: dict[int, list[str]] = defaultdict(list)
for label, text in zip(cluster_labels, texts, strict=True):
    cluster_docs[int(label)].append(text)


class_texts = [" ".join(cluster_docs[label]) for label in sorted(cluster_docs)]
X_class = vectorizer.transform(class_texts)
X_ctfidf = ctfidf.fit_transform(X_class).toarray()
print(f"c-TF-IDF matrix shape: {X_ctfidf.shape}  (one row per cluster)")
print("Sample c-TF-IDF values for first cluster:", X_ctfidf[0][:10])

#### Reading the c-TF-IDF output

Each float in `X_ctfidf` is a **c-TF-IDF score** for one (topic, term) pair. The matrix
has shape `(n_topics × vocab_size)`, so `X_ctfidf[i, j]` answers:

> *How much does term j distinguish topic i from all other topics?*

The score is computed as:

$$\text{c-TF-IDF}_{t,c} = \frac{f_{t,c}}{\sum_j f_{j,c}} \cdot \log\!\left(1 + \frac{A}{\sum_i f_{t,i}}\right)$$

Where:

- $f_{t,c}$ — frequency of term $t$ in the concatenated "class document" for cluster $c$
- $\sum_j f_{j,c}$ — total term count across that class document (normalizes for cluster size)
- $A$ — average number of words per class
- $\sum_i f_{t,i}$ — how many classes contain term $t$ at all (the IDF part)

In plain terms:

- **High positive score** → the term is frequent *within* this topic and rare *across other* topics — a strong topic keyword.
- **Near zero** → the term appears uniformly across topics; it doesn't distinguish anything.
- **Negative scores** are possible if the transformer subtracts a background frequency.

So the first 10 floats printed (`X_ctfidf[0][:10]`) are the c-TF-IDF weights for the
first 10 vocabulary terms in cluster 0. The terms with the *highest* values for each row
are what BERTopic surfaces as that topic's keywords.



**NOTE: BERTopic exposes the same c-TF-IDF scores through get_topic() — no need to
index into the raw matrix manually.**  

See below:**

`topic_model.get_topic(0)` 

### Step 5 — KeyBERTInspired: re-ranking keywords with embedding similarity

c-TF-IDF gives us statistically distinctive terms, but they can still be awkward
n-grams that are distinctive but not necesssarily representative of the topic. 
 **KeyBERTInspired** is a post-processing *representation model* that
re-ranks those candidates using embedding similarity, so the final keywords are
more semantically representative of the topic.

Reference: https://maartengr.github.io/BERTopic/getting_started/representation/representation.html

In [ ]:
from bertopic.representation import KeyBERTInspired

# Used for topic labeling (representation) and generating the
# topic_keyword_embeddings.
keybert_model = KeyBERTInspired(
    top_n_words=10,  # final words kept per topic
    nr_repr_docs=5,  # representative docs used per topic
    nr_samples=500,  # candidate docs sampled per topic
    nr_candidate_words=100,  # candidate words considered per topic
    random_state=RANDOM_SEED,  # reproducibility
)

### Step 6 — AI Labeling:  human-readable topic names

KeyBERTInspired gives us a ranked list of terms like `["rail transport", "high speed",
"california rail"]`. That is useful, but still jargon. The final stage passes those
keywords — plus a sample of representative documents — to an LLM which produces a
concise 3-word label such as **"California High-Speed Rail"**.

Note: Below we use Ollama, You can use any LLM backend to do this. The code is stored here: `src/ai_labeler.py` works


In [ ]:
from bertopic.representation import OpenAI as BertTopicOpenAI
from src.config import OLLAMA_MODEL, OLLAMA_URL
import httpx
from openai import OpenAI
import logging

logger = logging.getLogger(__name__)

# Wrapped in try/fail in case OLLAMA is not running
def ai_labeler(prompt: str) -> BertTopicOpenAI | None:
    """
    Return an OpenAI-based labeler if possible, otherwise None.
    """
    try:
        resp = httpx.get(OLLAMA_URL.replace("/v1", "/api/tags"), timeout=2)
        if resp.status_code == 200:
            client = OpenAI(base_url=OLLAMA_URL, api_key="ollama")
            return BertTopicOpenAI(
                client=client,  # OpenAI-compatible client
                model=OLLAMA_MODEL,  # LLM model name
                exponential_backoff=True,  # retry on rate limits
                chat=True,  # use chat completions API
                prompt=prompt,  # labeling prompt template
                nr_docs=5,  # representative docs per topic
            )
    except Exception:
        logger.warning("Could not connect to Ollama at %s. LLM-based labeling will be disabled.", OLLAMA_URL)
        return None

# BERTopic injects the topic's representative documents and keywords into the
# prompt, so we can just use placeholders here.
LABEL_PROMPT = """
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short but highly descriptive topic
label of at most 3 words. Make sure it is in the following format:
topic: <topic label>
"""

llm_model = ai_labeler(LABEL_PROMPT)


### Step 7 — Putting it all together: the full BERTopic pipeline

Below is the same pipeline as `src/topic_model.py:train_topic_model`, condensed into
one cell. You don't need to run it now (it's slow), but read through the parameter
block — every parameter you tune is one of these.

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from src.config import EMBEDDING_MODEL_NAME

# Set prior to training, but called after the model is fit.
print("Load embedding model...", EMBEDDING_MODEL_NAME)
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Ask BERTopic to use both KeyBERT and LLM-based labeling (if available) for topic representation.
representation_model = {
    "KeyBERT": keybert_model,
}
# LLM if backend available
if llm_model:
    representation_model["LLM"] = llm_model

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    ctfidf_model=ClassTfidfTransformer(),
    representation_model=representation_model,
    # Sets HDBSCAN's min_cluster_size. If not set above.
    # min_topic_size=10,  # Ignore.
    # Sets CountVectorizer's ngram_range. If not set above.
    # n_gram_range=(1, 1), # Ignore.
    top_n_words=10,
    # Required for topic evaluation
    calculate_probabilities=True,  # Do not change.
    verbose=True,  # Let's log!
)
topics, probabilities = topic_model.fit_transform(texts, embeddings)
print(f"Trained — found {topic_model.get_topic_info().shape[0] - 1} topics (plus -1 outlier).")

# 2. Evaluating the topic model

Three core inspection methods:

- `get_topic_info()` — one row per topic with size and label.
- `get_topic(topic_id)` — top words + c-TF-IDF scores for one topic.
- `get_document_info(texts)` — joins each document back to its topic assignment.

Topic `-1` is the **outlier bucket** — documents HDBSCAN couldn't confidently place.
These are not bad documents, they're just not part of any dense cluster.

In [ ]:
# Top-line summary of every topic
topic_model.get_topic_info().head(15)

In [ ]:
# Top n-grams for topic 0 with their c-TF-IDF scores
topic_model.get_topic(0)

In [ ]:
# How each document was labelled (truncate text for readability)
doc_info = topic_model.get_document_info(texts)
doc_info[["Document", "Topic", "Probability"]].head(10)

### Cross-reference with model artifacts

Once you run `uv run python -m src.topic_model`, the same information is dumped to
`output/`:

- `output/topic_assignments.csv` — `post_id`, `text`, `topic_id` for every doc.
- `output/topic_labels.json` — human-readable label per topic (LLM-generated when
  Ollama or OpenAI is available, falling back to the top-3 keywords).
- `output/topic_information.csv` — same as `get_topic_info()`.
- `output/topic_visualization.html` - see distance between topics and their
  density. (Open in browser)

These are what the demo app reads. And what you will use to evaluate changes to
the pipeline parameters after completing the exercise. 

In [ ]:
import pandas as pd

labels = json.loads((OUTPUT / "topic_labels.json").read_text())
assignments = pd.read_csv(OUTPUT / "topic_assignments.csv")
print("Topic labels (from disk):")
for tid, info in sorted(labels.items(), key=lambda kv: int(kv[0])):
    print(f"  {int(tid):3d}: {info['label']}")
print(f"\nAssignments dataframe ({len(assignments)} rows):")
assignments.head(10)

### Visualize the topic space

Two views worth knowing:

- **Intertopic distance map** (`visualize_topics`) — each circle is a topic, sized by
  the number of docs assigned. Topics close together in the map are semantically
  similar; widely-separated topics are well-distinguished. Outliers in the gaps
  between clusters are a clue that the model is finding *real* but loose structure.
- **Topic similarity heatmap** (`visualize_heatmap`) — pairwise cosine similarity
  between topic embeddings. Helpful when two topics are almost-but-not-quite the same.

Coherence vs diversity (slide 20):
- **Coherence** — do the words inside a topic actually go together? (high c-TF-IDF
  scores cluster.)
- **Diversity** — are the topics distinct from each other? (low off-diagonal in the
  heatmap.)

In [ ]:
# Inline if running interactively. The same chart is also dumped to
# output/topic_visualization.html by src/topic_model.py.
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_heatmap()

# Exercise 

Time: ~10-15 minutes coding; 2-10 minutes runtime

After running this notebook, update the code in `src/topic_model.py` 

Objective:  Tune the model to find the 7 distinct topics created by generator_posts.py.

Pair up with a partner. Work together to adjust the parameters to each
other's models. To save time training and learn from each other, try to test
different changes on each other's models. 

1. Open [`src/topic_model.py` ~line 154](../src/topic_model.py#L154).
3. Run `uv run pytest -k test_topic_model` to test your implementation.
2. Retrain the model by running: 

   ```
   uv run python -m src.topic_model
   ```
   Warning: This takes 2-10 minutes. 
   **See "Run the code" below to understand what to expect in this step.**

4. Review the output of the model in output/
5. Open the Demo App and see the topic results. 
6. Review and evaluate the results of each persons changes together. Did the
   model run? Or did it break? If it broke, why did it break? How did the
   evaluation metrics change?
7. Report out your findings to the group (optional)

Note: We recognize training the topic model may take some time, do not worry if
you are unable to complete this exercise in the time available. You will not fall
behind. 

See reference implementations in `solutions/topic_model.py` if you get stuck.

# Run the code

Now train BERTopic end-to-end using the embeddings you preprocessed in notebook 2:

```bash
uv run python -m src.topic_model
```

> Runtime: another 2-10 minutes, with most of the time spent labeling topics locally.

What to look for:

| Line | Meaning |
|---|---|
| `Loaded 154 posts from disk.` | Preprocessing output was found. |
| `Training on 154 posts.` | Embeddings are present. |
| `Training on 0 posts.` followed by a type error about strings | Embeddings are missing or empty; fix `generate_embeddings` and rerun preprocessing. |
| BERTopic stages such as `Dimensionality -> Cluster -> Representation` | Normal training flow. |
| The `Representation` progress bar moving from `0/8` to `8/8` | The labeling step is running; this is typically the slowest part. |

### Optional: run from inside the notebook

If you prefer not to switch to a terminal, you can run the same preprocessing
pipeline here using your edited `src/topic_model.py`. 

Expect a 2-10 minute run and watch for warnings such as "No valid topics found — all documents assigned to outlier class."

In [ ]:
# Run the full BERTtopic training pipeline.
from src.topic_model import TopicModeler
TopicModeler().run()